# Phase 2: can a linear probe read the act/refrain decision, and when?

**Question.** At which layers, and at which moment (notice vs commit), can a linear classifier tell "about to blackmail" from "about to hold back" better than chance?

**Method.**
- One probe per layer: standardise, then L2-regularised logistic regression (`C=0.05`), scored by 5-fold cross-validated AUROC.
- A **permutation null**: shuffle the labels 50 times and rerun everything, so a lucky-looking layer is compared with what chance produces when you test 48 layers.
- Two bars: a **pre-registered window** (layers 6-12, chosen before looking) and the stricter **all-48-layer** bar (95th percentile of the max over layers).
- Also the **difference-of-means direction** per layer (`commit_diff_means.pt`), which Phases 3-5 steer with.

| | |
|---|---|
| Needs | CPU only |
| Reads | `contrast_acts.pt`, `contrast_acts_commit.pt` (from Phase 1) |
| Writes | `commit_diff_means.pt`, `results/probe_*.png` |

**Running on its own Colab.** This notebook shares nothing with the others except one Google Drive folder (`MyDrive/gemma-interp`, set in the setup cell, the same in every notebook).
It reads the files it needs from that folder and writes what it makes there, so the next notebook can pick it up even though it runs in a different Colab.
If a small input file is missing and `REPO_RAW` is set (the raw URL of your public GitHub folder), it is downloaded automatically; otherwise you get a message naming the file and the notebook that creates it.
Use a GPU runtime for every notebook except phase 2 (CPU is enough). Nothing is kept on the Colab's own disk, so a disconnect loses nothing that was already saved to Drive.

In [ ]:
import os, json, time, numpy as np
from collections import defaultdict
from functools import partial
# Each notebook can run in its own Colab, and a Colab disk is wiped when it disconnects. So everything that must survive
# or be handed to another notebook (activations, directions, saved samples) lives in a Google Drive folder.
USE_DRIVE  = True                                                 # set False to use the local folder instead
DRIVE_DIR  = "MyDrive/gemma-interp"                               # the same Drive folder in every notebook
REPO_RAW   = ""                                                   # optional: raw GitHub folder holding the small input files, e.g.
                                                                  # "https://raw.githubusercontent.com/<user>/<repo>/main/gemma-interp"
try:
    import google.colab; ON_COLAB = True
except ImportError:
    ON_COLAB = False
if USE_DRIVE and ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    BASE = f"/content/drive/{DRIVE_DIR}"
else:
    BASE = os.environ.get("GEMMA_BASE", os.getcwd())
OUT = f"{BASE}/results"; os.makedirs(OUT, exist_ok=True)           # everything generated lands here
print("working folder:", BASE)

def need(fname, optional=False):                                  # make sure BASE/fname exists: use it, else download it, else say what to do
    path = f"{BASE}/{fname}"
    if os.path.exists(path): return path
    if REPO_RAW:
        import urllib.request
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(f"{REPO_RAW}/{fname}", path); print("downloaded", fname); return path
        except Exception as e: print(f"could not download {fname}: {e}")
    if optional: print(f"(optional file {fname} not found - that is fine)"); return None
    raise FileNotFoundError(f"{fname} is not in {BASE}. Put it in that Drive folder (the notebook that creates it is named in README.md), "
                            f"or set REPO_RAW to the raw GitHub folder that holds it.")
SEED, N_SCEN, MAX_NEW = 0, 15, 1200
GRADER_MODEL = os.environ.get("GRADER_MODEL", "gemini-3.5-flash") # Gemini model that grades outputs; change it if your key says "model not found"
# Keys are never written in a notebook. ensure_key() finds one in the environment, then in Colab Secrets,
# and otherwise asks you to type it (hidden, kept only for this session):
#   GEMINI_API_KEY  (grading)      HF_TOKEN  (first download of Gemma; accept the licence on Hugging Face first)
def ensure_key(name):
    if os.environ.get(name): return
    try:
        from google.colab import userdata                         # Colab Secrets, if you are in Colab
        os.environ[name] = userdata.get(name); return
    except Exception: pass
    import getpass
    os.environ[name] = getpass.getpass(f"{name} (typed here, not saved in the notebook): ")

In [ ]:
import torch, matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

N_PERM = 50
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
def probe(): return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=2000))
def auroc(X, y): return cross_val_score(probe(), X, y, cv=skf, scoring="roc_auc", n_jobs=-1).mean()

def probe_curve(fname, tag, window=None):
    d = torch.load(need(fname))                                   # made by phase 1, in the same Drive folder
    act, ref = d["act"].float().numpy(), d["ref"].float().numpy()
    y = np.r_[np.ones(len(act)), np.zeros(len(ref))]; nl = act.shape[1]
    Xs = [np.concatenate([act[:, L], ref[:, L]], 0) for L in range(nl)]
    real = np.array([auroc(Xs[L], y) for L in range(nl)])
    rng = np.random.default_rng(0); null = np.empty((N_PERM, nl))
    for p in range(N_PERM):                                       # a fresh shuffle every iteration
        yp = rng.permutation(y)
        for L in range(nl): null[p, L] = auroc(Xs[L], yp)
    thr_full = np.percentile(null.max(axis=1), 95)                # corrects for testing all layers
    print(f"[{tag}] act={len(act)} ref={len(ref)} | peak AUROC {real.max():.3f} @L{real.argmax()}")
    print(f"[{tag}] all-layer bar {thr_full:.3f} -> clears layers {np.where(real > thr_full)[0].tolist()}")
    if window:
        thr_win = np.percentile(null[:, list(window)].max(axis=1), 95)
        print(f"[{tag}] pre-registered L{window[0]}-{window[-1]} bar {thr_win:.3f} -> clears "
              f"{[L for L in window if real[L] > thr_win]}")
    lo, _, hi = np.percentile(null, [2.5, 50, 97.5], axis=0)
    fig, ax = plt.subplots(figsize=(9, 5)); L = np.arange(nl)
    ax.plot(L, real, label="probe AUROC"); ax.fill_between(L, lo, hi, color="gray", alpha=0.3, label="permutation null (95%)")
    ax.axhline(thr_full, color="tab:red", ls="--", lw=1, label=f"all-layer bar {thr_full:.3f}")
    if window: ax.axvspan(window[0]-0.5, window[-1]+0.5, color="tab:green", alpha=0.07, label="pre-registered window")
    ax.axhline(0.5, c="k", lw=0.8, ls=":"); ax.set_ylim(0.4, 0.8); ax.legend(fontsize=8)
    ax.set_xlabel("layer"); ax.set_ylabel("AUROC"); ax.set_title(f"{tag} anchor: probe vs permutation null")
    plt.tight_layout(); plt.savefig(f"{OUT}/probe_{tag}.png", dpi=150); plt.show()
    return real

## 1. Moment 1: the model has just noticed the leverage
Expected (original run): nothing clears the bar. The peak was about 0.60 AUROC, below the all-layer threshold of about 0.65.

In [ ]:
real_notice = probe_curve("contrast_acts.pt", "notice")

## 2. Moment 2: the end of the reasoning, just before acting
Expected (original run): peak about 0.74 AUROC at layer 19. Inside the pre-registered window (6-12) only layers 10-11 (about 0.67) clear the bar;
layers 12-16 sit right at the all-layer line, so "readable from layer 10 onward" is generous. The high layers clear it comfortably.

In [ ]:
real_commit = probe_curve("contrast_acts_commit.pt", "commit", window=range(6, 13))

## 3. Is the probe method itself sound? (planted-signal check)
Plant a known signal of strength `delta` (in standard deviations) along a random direction and confirm the probe recovers it:
AUROC should be about 0.5 at `delta=0` and climb as `delta` grows. If it does not, a null result on the real data would mean nothing.

In [ ]:
d = torch.load(need("contrast_acts.pt")); act, ref = d["act"].float().numpy(), d["ref"].float().numpy()
y = np.r_[np.ones(len(act)), np.zeros(len(ref))]
L = 9; X = np.concatenate([act[:, L], ref[:, L]], 0); sd = X.std(0)
for delta in [0.0, 0.5, 1.0, 2.0, 4.0]:
    a = []
    for s in range(5):
        r = np.random.default_rng(s); yc = r.permutation(y)
        v = r.standard_normal(X.shape[1]); v /= np.linalg.norm(v)
        a.append(auroc(X + delta * (yc == 1)[:, None] * (v * sd), yc))
    print(f"delta={delta}: AUROC={np.mean(a):.3f}")

## 4. The steering direction: mean(act) minus mean(refrain), per layer (commit anchor)

In [ ]:
d = torch.load(need("contrast_acts_commit.pt"))
dirs = d["act"].float().mean(0) - d["ref"].float().mean(0)       # [48, 3840]
torch.save(dirs, f"{BASE}/commit_diff_means.pt")
print(dirs.shape, "| norm at L19:", round(dirs[19].norm().item(), 1))

## Reading the result
- The decision is **not** readable when the leverage is first noticed; it becomes readable by the end of the reasoning. The model seems to settle as it deliberates.
- **Caution before building on it.** The commit anchor is the last token of text the model has already written, so the probe may be reading the content of that plan rather than an internal decision.
  A text-only baseline (a classifier on the reasoning text) would test that; it has not been run.
- 0.74 AUROC is modest as a monitor signal, and the sets are small (about 70-100 per class).